# Stock Price Prediction using LSTM with PyTorch
## Predicting stock prices using historical data and deep learning

### Step 0: Install Required Libraries
Run this cell if you haven't installed these packages yet:

In [ ]:
# !pip install torch torchvision torchaudio
# !pip install pandas numpy scikit-learn matplotlib yfinance

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
from sklearn.preprocessing import MinMaxScaler
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Step 2: Data Loading and Preprocessing

In [ ]:
def load_stock_data(ticker='AAPL', days=500):
    """Load stock data from Yahoo Finance"""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    print(f"Downloading {ticker} data from {start_date.date()} to {end_date.date()}...")
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    print(f"Downloaded {len(data)} trading days")
    
    return data['Close'].values.reshape(-1, 1)

# Load data
TICKER = 'AAPL'  # Change this to any stock ticker you want
raw_data = load_stock_data(TICKER, days=500)
print(f"Data shape: {raw_data.shape}")
print(f"Price range: ${raw_data.min():.2f} - ${raw_data.max():.2f}")

### Step 3: Normalize Data

In [ ]:
# Normalize data to range [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(raw_data)

print(f"Original data min/max: {raw_data.min():.2f} / {raw_data.max():.2f}")
print(f"Scaled data min/max: {scaled_data.min():.6f} / {scaled_data.max():.6f}")

# Visualize raw data
plt.figure(figsize=(12, 4))
plt.plot(raw_data, label='Stock Price')
plt.title(f'{TICKER} Historical Stock Price')
plt.xlabel('Trading Days')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Step 4: Create Sequences

In [ ]:
def create_sequences(data, seq_length=30):
    """Create sequences for LSTM training
    X: past seq_length days
    y: next day price
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 30  # Use past 30 days to predict next day
X, y = create_sequences(scaled_data, SEQ_LENGTH)

print(f"Sequences created:")
print(f"  X shape: {X.shape} (samples, sequence_length, features)")
print(f"  y shape: {y.shape} (samples, 1)")
print(f"  Total samples: {len(X)}")

### Step 5: Split Data into Train and Test Sets

In [ ]:
# Train-test split (80-20, respecting temporal order)
TEST_SPLIT = 0.2
split_idx = int(len(X) * (1 - TEST_SPLIT))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train).to(device)
y_train = torch.FloatTensor(y_train).to(device)
X_test = torch.FloatTensor(X_test).to(device)
y_test = torch.FloatTensor(y_test).to(device)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Split date: sample {split_idx} out of {len(X)}")

### Step 6: Define LSTM Model

In [ ]:
class StockPriceLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2, output_size=1):
        super(StockPriceLSTM, self).__init__()
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Take the last output
        last_hidden = lstm_out[:, -1, :]
        
        # Pass through fully connected layer
        output = self.fc(last_hidden)
        
        return output

# Initialize model
model = StockPriceLSTM(input_size=1, hidden_size=50, num_layers=2, output_size=1).to(device)
print(model)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters())}")

### Step 7: Define Loss Function and Optimizer

In [ ]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

### Step 8: Train the Model

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

train_losses = []
test_losses = []

print("Starting training...\n")
for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for i in range(0, len(X_train), BATCH_SIZE):
        batch_X = X_train[i:i+BATCH_SIZE]
        batch_y = y_train[i:i+BATCH_SIZE]
        
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= (len(X_train) // BATCH_SIZE)
    train_losses.append(train_loss)
    
    # Evaluation phase
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test)
        test_loss = criterion(test_pred, y_test).item()
        test_losses.append(test_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

print("\nTraining completed!")

### Step 9: Evaluate the Model

In [ ]:
# Get predictions
model.eval()
with torch.no_grad():
    train_predictions = model(X_train).cpu().numpy()
    test_predictions = model(X_test).cpu().numpy()
    train_actual = y_train.cpu().numpy()
    test_actual = y_test.cpu().numpy()

# Inverse transform to get actual prices
train_pred_real = scaler.inverse_transform(train_predictions)
train_actual_real = scaler.inverse_transform(train_actual)
test_pred_real = scaler.inverse_transform(test_predictions)
test_actual_real = scaler.inverse_transform(test_actual)

# Calculate RMSE
train_rmse = np.sqrt(np.mean((train_pred_real - train_actual_real) ** 2))
test_rmse = np.sqrt(np.mean((test_pred_real - test_actual_real) ** 2))

print("=" * 50)
print("Model Evaluation Results")
print("=" * 50)
print(f"Train RMSE: ${train_rmse:.2f}")
print(f"Test RMSE:  ${test_rmse:.2f}")
print(f"\nTest data price range: ${test_actual_real.min():.2f} - ${test_actual_real.max():.2f}")
print(f"RMSE as % of price range: {(test_rmse / (test_actual_real.max() - test_actual_real.min())) * 100:.2f}%")

### Step 10: Visualize Results

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Training Loss
axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0, 0].plot(test_losses, label='Test Loss', linewidth=2)
axes[0, 0].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training Data Predictions
axes[0, 1].plot(train_actual_real, label='Actual Price', linewidth=2, alpha=0.8)
axes[0, 1].plot(train_pred_real, label='Predicted Price', linewidth=2, alpha=0.8)
axes[0, 1].set_title('Training Data: Predicted vs Actual', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Time Step')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Test Data Predictions
axes[1, 0].plot(test_actual_real, label='Actual Price', linewidth=2, alpha=0.8)
axes[1, 0].plot(test_pred_real, label='Predicted Price', linewidth=2, alpha=0.8)
axes[1, 0].set_title('Test Data: Predicted vs Actual', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Error Distribution
train_error = (train_actual_real - train_pred_real).flatten()
test_error = (test_actual_real - test_pred_real).flatten()
axes[1, 1].hist(train_error, bins=30, alpha=0.6, label='Train Error')
axes[1, 1].hist(test_error, bins=30, alpha=0.6, label='Test Error')
axes[1, 1].set_title('Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Error ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Step 11: Analyze Prediction Patterns

In [ ]:
# Calculate statistics
print("\n" + "=" * 50)
print("Detailed Analysis")
print("=" * 50)

# Direction accuracy
train_direction = np.sign(np.diff(train_actual_real.flatten()))
train_pred_direction = np.sign(np.diff(train_pred_real.flatten()))
train_direction_acc = np.mean(train_direction == train_pred_direction) * 100

test_direction = np.sign(np.diff(test_actual_real.flatten()))
test_pred_direction = np.sign(np.diff(test_pred_real.flatten()))
test_direction_acc = np.mean(test_direction == test_pred_direction) * 100

print(f"\nDirection Prediction Accuracy:")
print(f"  Train: {train_direction_acc:.2f}%")
print(f"  Test: {test_direction_acc:.2f}%")
print(f"  (50% = random guess)")

# Average prediction error
print(f"\nAverage Absolute Error:")
print(f"  Train: ${np.mean(np.abs(train_error)):.2f}")
print(f"  Test: ${np.mean(np.abs(test_error)):.2f}")

### Step 12: Make Predictions for Future Dates

In [ ]:
# Predict the next day price
last_sequence = scaled_data[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
last_sequence = torch.FloatTensor(last_sequence).to(device)

model.eval()
with torch.no_grad():
    next_pred_scaled = model(last_sequence).cpu().numpy()
    next_pred_real = scaler.inverse_transform(next_pred_scaled)[0][0]

last_price = raw_data[-1][0]
price_change = next_pred_real - last_price
price_change_pct = (price_change / last_price) * 100

print("\n" + "=" * 50)
print("Next Day Prediction")
print("=" * 50)
print(f"Last actual price: ${last_price:.2f}")
print(f"Predicted next day: ${next_pred_real:.2f}")
print(f"Predicted change: ${price_change:.2f} ({price_change_pct:+.2f}%)")
print("\n⚠️  Warning: This prediction should NOT be used for trading decisions.")
print("Market conditions, news, and external factors can dramatically affect prices.")

### Step 13: Save the Model

In [ ]:
# Save model
model_path = f'lstm_model_{TICKER}.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

# Save scaler for future use
import pickle
scaler_path = f'scaler_{TICKER}.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to: {scaler_path}")

---

## Summary & Limitations

### What We Built:
- 2-layer LSTM model with 50 hidden units
- Predicts next-day stock price using past 30 days
- Trained on normalized historical data

### Important Limitations:
1. **Lag Effect**: The model tends to follow past trends with a delay
2. **No External Factors**: Only uses historical prices, ignores news, earnings, market conditions
3. **Data Leakage**: If not careful with train-test split, can learn from future data
4. **Overfitting**: LSTM can memorize noise in historical patterns
5. **Market Efficiency**: If the pattern was profitable, the market would have already corrected it

### Use Cases:
- ✅ Learning deep learning concepts
- ✅ Understanding time series modeling
- ❌ Making actual trading decisions
- ❌ Replacing professional financial advice